In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV  #***
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, f1_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor

from tqdm import tqdm  # Provides the progress of model running
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
concrete = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\Concrete_Data.csv")
X , y = concrete.drop(['Strength'], axis=1), concrete['Strength']

In [3]:
kfold = KFold(n_splits=5, shuffle=True, random_state=26)

In [4]:
alphas = np.linspace(0.001, 10, 30)
params = {'alpha':alphas}
ridge = Ridge()
gcv = GridSearchCV(estimator=ridge, param_grid=params, cv=kfold)
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009378,0.000878,0.006476,0.000465,0.001000,{'alpha': 0.001},0.614744,0.566049,0.593351,0.611864,0.628377,0.602877,0.021535,30
1,0.009261,0.000826,0.005926,0.000932,0.345793,{'alpha': 0.3457931034482759},0.614744,0.566049,0.593351,0.611864,0.628377,0.602877,0.021536,29
2,0.008242,0.000790,0.005868,0.001370,0.690586,{'alpha': 0.6905862068965518},0.614744,0.566049,0.593351,0.611865,0.628377,0.602877,0.021536,28
3,0.008482,0.000804,0.005326,0.001210,1.035379,{'alpha': 1.0353793103448277},0.614744,0.566049,0.593351,0.611865,0.628377,0.602877,0.021536,27
4,0.012324,0.001009,0.008755,0.001754,1.380172,{'alpha': 1.3801724137931035},0.614744,0.566048,0.593351,0.611865,0.628377,0.602877,0.021536,26


In [5]:
gcv.best_params_, gcv.best_score_

({'alpha': np.float64(10.0)}, np.float64(0.6028781702961167))

In [6]:
alphas = np.linspace(0.001, 10, 30)
ratios = np.linspace(0.001,1,10) 

params = {'alpha':alphas,'l1_ratio':ratios}

el = ElasticNet()
gcv = GridSearchCV(estimator=el, param_grid=params, cv=kfold)
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head()
gcv.best_params_, gcv.best_score_

({'alpha': np.float64(2.759344827586207), 'l1_ratio': np.float64(0.001)},
 np.float64(0.6029818143707282))

In [7]:
params = {'alpha':np.linspace(0.001, 10, 30), 'l1_ratio':np.linspace(0.001,1,10)}

el = ElasticNet()
gcv = GridSearchCV(estimator=el, param_grid=params, cv=kfold)
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head()
gcv.best_params_, gcv.best_score_

({'alpha': np.float64(2.759344827586207), 'l1_ratio': np.float64(0.001)},
 np.float64(0.6029818143707282))

In [8]:
params = {'alpha':np.linspace(0.001, 10, 30), 'l1_ratio':np.linspace(0.001,1,10)}

el = ElasticNet()
gcv = GridSearchCV(estimator=el, param_grid=params, cv=kfold, scoring='neg_root_mean_squared_error')
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head()
gcv.best_params_, gcv.best_score_

({'alpha': np.float64(3.104137931034483), 'l1_ratio': np.float64(0.001)},
 np.float64(-10.456383568226837))

In [9]:
features = [2,3,4,5,6,7]
params = {'max_features':features}
rf = RandomForestRegressor(random_state=26)

gcv = GridSearchCV(estimator=rf, param_grid=params, cv=kfold, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head
gcv.best_params_, gcv.best_score_

({'max_features': 4}, np.float64(-4.751942056038116))

In [10]:
features = [2,3,4,5,6,7]
params = {'max_features':features}
rf = RandomForestRegressor(random_state=26)

gcv = GridSearchCV(estimator=rf, param_grid=params, cv=kfold, verbose=3, scoring='neg_root_mean_squared_error')  # verbose=1 gives information (max=3)
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head
gcv.best_params_, gcv.best_score_

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV 1/5] END ...................max_features=2;, score=-6.111 total time=   1.0s
[CV 2/5] END ...................max_features=2;, score=-4.679 total time=   1.0s
[CV 3/5] END ...................max_features=2;, score=-4.431 total time=   1.0s
[CV 4/5] END ...................max_features=2;, score=-6.111 total time=   0.9s
[CV 5/5] END ...................max_features=2;, score=-4.112 total time=   0.9s
[CV 1/5] END ...................max_features=3;, score=-5.645 total time=   1.2s
[CV 2/5] END ...................max_features=3;, score=-4.140 total time=   1.1s
[CV 3/5] END ...................max_features=3;, score=-4.298 total time=   0.6s
[CV 4/5] END ...................max_features=3;, score=-5.822 total time=   0.2s
[CV 5/5] END ...................max_features=3;, score=-4.067 total time=   0.3s
[CV 1/5] END ...................max_features=4;, score=-5.565 total time=   0.3s
[CV 2/5] END ...................max_features=4;, 

({'max_features': 4}, np.float64(-4.751942056038116))

In [11]:
# n_jobs
features = [2,3,4,5,6,7]
params = {'max_features':features}
rf = RandomForestRegressor(random_state=26)

gcv = GridSearchCV(estimator=rf, param_grid=params, cv=kfold, verbose=3, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
pd_cv.head
gcv.best_params_, gcv.best_score_

Fitting 5 folds for each of 6 candidates, totalling 30 fits


({'max_features': 4}, np.float64(-4.751942056038116))

#  Scaling + KNN

In [12]:
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [13]:
pipe = Pipeline([('scaler',StandardScaler()),('KNN', KNeighborsRegressor())])
params = {'scaler':[None, StandardScaler(), MinMaxScaler()],
         'KNN__n_neighbors':[1,2,3,4,5,6]}

gcv = GridSearchCV(estimator=pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'KNN__n_neighbors': 3, 'scaler': StandardScaler()},
 np.float64(-8.737696015609808))

In [14]:
std_scl = StandardScaler()
mm = MinMaxScaler()
knn = KNeighborsRegressor()

pipe = Pipeline([('SCL', std_scl),('KNN',knn)])
params = {'SCL':[None, std_scl, mm],
         'KNN__n_neighbors':[1,2,3,4,5,6]}

gcv = GridSearchCV(estimator=pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'KNN__n_neighbors': 3, 'SCL': StandardScaler()},
 np.float64(-8.737696015609808))

In [16]:
tst = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\testConcrete.csv")
tst

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210
5,150,23,23,114,4,883,638,36
6,480,64,0,292,3,896,776,180
7,393,49,82,132,1,887,830,271
8,284,63,1,138,1,804,725,44
9,206,38,0,103,2,818,719,191


### Inferencing

In [17]:
# best model way
knn = KNeighborsRegressor(n_neighbors=3)
scl=StandardScaler()
bm = Pipeline([('SCL', scl),('KNN',knn)])
bm.fit(X,y)
bm.predict(tst)

array([57.69333333, 48.86      , 31.04      , 51.60333333, 56.81      ,
       37.76      , 52.05666667, 39.45      , 55.62      , 54.84666667,
       24.09333333, 67.06666667, 52.05666667, 43.24666667])

In [18]:
# using GCV
gcv.predict(tst)

array([57.69333333, 48.86      , 31.04      , 51.60333333, 56.81      ,
       37.76      , 52.05666667, 39.45      , 55.62      , 54.84666667,
       24.09333333, 67.06666667, 52.05666667, 43.24666667])

In [19]:
bm = gcv.best_estimator_
bm.predict(tst)

array([57.69333333, 48.86      , 31.04      , 51.60333333, 56.81      ,
       37.76      , 52.05666667, 39.45      , 55.62      , 54.84666667,
       24.09333333, 67.06666667, 52.05666667, 43.24666667])

# Housing Dataset

In [20]:
housing = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\Housing.csv")
X , y = housing.drop(['price'], axis=1), housing['price']

In [21]:
ohe=OneHotEncoder(drop="first")

trans = ColumnTransformer(transformers=[("OHE", ohe, make_column_selector(dtype_include=object))],     
                         remainder="passthrough", verbose_feature_names_out=False)

In [22]:
rf = RandomForestRegressor(random_state=26 )
pipe = Pipeline([('OHE',trans), ('RF',rf)])
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 26,
 'verbose': 0,
 'warm_start': False}

In [23]:
pipe.get_params()

{'memory': None,
 'steps': [('OHE', ColumnTransformer(remainder='passthrough',
                     transformers=[('OHE', OneHotEncoder(drop='first'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000002A78E351A90>)],
                     verbose_feature_names_out=False)),
  ('RF', RandomForestRegressor(random_state=26))],
 'transform_input': None,
 'verbose': False,
 'OHE': ColumnTransformer(remainder='passthrough',
                   transformers=[('OHE', OneHotEncoder(drop='first'),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000002A78E351A90>)],
                   verbose_feature_names_out=False),
 'RF': RandomForestRegressor(random_state=26),
 'OHE__force_int_remainder_cols': 'deprecated',
 'OHE__n_jobs': None,
 'OHE__remainder': 'passthrough',
 'OHE__sparse_threshold': 0.3,
 'OHE__transformer_weights': None,
 'OHE__transformers': [('OHE',
   OneHotEncoder(drop='

In [24]:
# features = [2,3,4,5,6,7]
# params = {'max_features':features} # cause error due to in pipeline max_features is labeled as RF__max_features

features = [2,3,4,5,6,7]
params = {'RF__max_features':features}

gcv = GridSearchCV(estimator= pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'RF__max_features': 4}, np.float64(-1098024.9290877976))

# Chemical Process - Imputing

In [25]:
chem = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\ChemicalProcess.csv")
chem

,Yield,BiologicalMaterial01,BiologicalMaterial02,BiologicalMaterial03,BiologicalMaterial04,BiologicalMaterial05,BiologicalMaterial06,BiologicalMaterial07,BiologicalMaterial08,BiologicalMaterial09,...,ManufacturingProcess36,ManufacturingProcess37,ManufacturingProcess38,ManufacturingProcess39,ManufacturingProcess40,ManufacturingProcess41,ManufacturingProcess42,ManufacturingProcess43,ManufacturingProcess44,ManufacturingProcess45
0,38.00,6.25,49.58,56.97,12.74,19.51,43.73,100.0,16.66,11.44,...,0.019,0.5,3,7.2,NaN,NaN,11.6,3.0,1.8,2.4
1,42.44,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.019,2.0,2,7.2,0.1,0.15,11.1,0.9,1.9,2.2
2,42.03,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.018,0.7,2,7.2,0.0,0.00,12.0,1.0,1.8,2.3
3,41.42,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,...,0.018,1.2,2,7.2,0.0,0.00,10.6,1.1,1.8,2.1
4,42.49,7.47,63.33,72.25,14.02,17.91,54.66,100.0,18.22,12.80,...,0.017,0.2,2,7.3,0.0,0.00,11.0,1.1,1.7,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,39.66,6.71,56.32,66.19,12.35,20.02,50.26,100.0,17.54,12.50,...,NaN,2.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
172,39.68,6.87,56.74,66.61,12.55,20.18,50.80,100.0,17.48,12.41,...,NaN,1.0,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
173,42.23,7.50,58.41,68.30,13.33,20.81,52.96,100.0,17.23,12.04,...,NaN,1.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
174,38.48,7.53,58.36,69.25,14.35,20.57,51.31,100.0,17.87,12.77,...,NaN,2.3,0,0.0,0.0,0.00,0.0,0.5,0.0,0.0


In [26]:
X, y = chem.drop('Yield', axis=1) , chem['Yield']

In [27]:
imp = SimpleImputer()
rf = RandomForestRegressor(random_state=26)
pipe = Pipeline([('IMP',imp),('MODEL',rf)])

params = {'IMP__strategy':['mean', 'median'],
         'MODEL__max_features':[2,5,10,15,25,30]}

gcv = GridSearchCV(estimator= pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'IMP__strategy': 'median', 'MODEL__max_features': 30},
 np.float64(-1.1026080480472165))

In [28]:
imp = SimpleImputer(strategy='mean')
rf = RandomForestRegressor(random_state=26)
pipe = Pipeline([('IMP',imp),('MODEL',rf)])

params = {'MODEL__max_features':[2,5,10,15,25,30]}

gcv = GridSearchCV(estimator= pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'MODEL__max_features': 25}, np.float64(-1.1101464968009211))

In [29]:
imp = SimpleImputer(strategy='median')
rf = RandomForestRegressor(random_state=26)
pipe = Pipeline([('IMP',imp),('MODEL',rf)])

params = {'MODEL__max_features':[2,5,10,15,25,30]}

gcv = GridSearchCV(estimator= pipe, param_grid=params, cv=kfold, n_jobs=-1, scoring='neg_root_mean_squared_error') 
gcv.fit(X, y)
pd_cv = pd.DataFrame(gcv.cv_results_)
gcv.best_params_, gcv.best_score_

({'MODEL__max_features': 30}, np.float64(-1.1026080480472165))